
# Demo: Create Supervisor Agent in Databricks UI

[Create a multi-agent supervisor system](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/multi-agent-supervisor#create-a-multi-agent-supervisor-system)


## Agents ➡️ Create Agent ➡️ Supervisor Agent

In [0]:
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
print(f"Go to https://{workspace_url}/ml/bricks")

![](./databricks-ui-agents.png)


Click the **Create Agent** button in the top right.

You should see the following screen where you find the **Supervisor Agent** tile.

![](./databricks-ui-create-new-agent.png)

## Demo Supervisor Agent

![](./databricks-ui-demo-supervisor-agent.png)

## Execute Supervisor Agent


Click **Endpoint** button on your right-hand side.

![](./databricks-ui-demo-supervisor-agent-endpoint.png)


Click **Use** button.

Click **Get code** drop-down menu.

![](./databricks-ui-get-code-python-api.png)

In [0]:
%pip install -U databricks-sdk mlflow-skinny[databricks]>=3.12 databricks-openai>=0.15.0
%restart_python

In [0]:
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
host = ws.config.host
print(host)

### Enable MLflow Tracing

The following cells use **OpenAI Responses API** to interact with the supervisor agent.

It is recommended to enable [MLflow Tracing](https://docs.databricks.com/aws/en/mlflow3/genai/tracing) when developing GenAI applications.

Tracing provides a step-by-step analysis of your app's execution, helping you debug latency, cost, and quality issues.

Traces can be used with Agent Evaluation to [measure your app's quality](https://docs.databricks.com/aws/en/generative-ai/agent-evaluation/evaluate-agent), and the same Trace will be [automatically captured](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent#agent-enhanced-inference-tables) once you deploy your application.

In [0]:
import mlflow

mlflow.openai.autolog()

In [0]:
import re


endpoints = ws.serving_endpoints.list()

current_user = ws.current_user.me().user_name
supervisor_endpoint = next(
    (
        ep for ep in endpoints
        if re.match(r"mas-\w+-endpoint", ep.name)
        and ep.creator == current_user
    ),
    None
)
if supervisor_endpoint_name := supervisor_endpoint.name:
    print(f"Supervisor Agent Serving Endpoint: {supervisor_endpoint_name}")
else:
    dbutils.notebook.exit('Supervisor Agent Serving Endpoint not found')

In [0]:
supervisor_endpoint

In [0]:
from openai import OpenAI

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

base_url=f"{host}/serving-endpoints"
model = supervisor_endpoint_name

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=base_url
)

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": "What is a supervisor agent in Databricks?"
        }
    ]
)

### Display Output

In [0]:
# https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html
from IPython.display import Markdown, display


output_markdown = " ".join(
    getattr(content, "text", "")
    for output in response.output
    if getattr(output, "type", None) == "message"
    for content in (getattr(output, "content", None) or [])
)
display(Markdown(output_markdown))